# Knowledge Graphs

A self-contained refresher on **knowledge graphs (KGs)** — modelling a domain as a network of
entities and the typed relationships between them, then querying and reasoning over that network.
The runnable parts use **rdflib** (RDF triples + SPARQL) and **networkx** (graph algorithms).

**Domain:** Symbolic AI & Logic  ·  **runnable:** yes (`pip install rdflib networkx`)

> Related notebooks: [`rdflib`](rdflib.ipynb) and [`sparql`](sparql.ipynb) drill into the RDF
> serialization and query language; [`owl`](owl.ipynb) covers ontology semantics and reasoners;
> [`knowledge-graph-embeddings`](../02-ai-ml-tooling/knowledge-graph-embeddings.ipynb) covers the
> vector/ML side. This notebook is the *concept* glue that ties them together.

## 1. What & Why

A **knowledge graph** represents knowledge as a graph: **nodes are entities** (people, places,
products, concepts) and **edges are typed, directed relationships** between them
(`Alice —worksAt→ Acme`). A *schema* or *ontology* layered on top says which entity types and
relation types are allowed and what they mean.

The problem it solves: real-world knowledge is **highly connected and irregular**. Relational
tables force you to pick a rigid schema up front and pay expensive joins to follow relationships;
documents bury the structure in prose. A KG makes the *relationships first-class* — you store and
traverse them directly, mix heterogeneous facts in one model, and add new relation types without a
migration.

**Reach for a KG when:**
- The questions are about *connections* and multi-hop paths ("which suppliers of my suppliers are
  in a sanctioned country?") more than aggregates over uniform rows.
- Data is heterogeneous and evolving — you're integrating many sources with overlapping entities.
- You want machine-readable semantics: shared vocabularies, inference, and explainable answers.
- You're grounding an LLM (GraphRAG): retrieve a relevant subgraph instead of loose text chunks.

**Don't bother when** your data is tabular and the queries are aggregations/analytics (use SQL or a
columnar store), or when there are essentially no relationships to traverse. A graph database that
only ever does single-hop lookups is a slow key-value store.

## 2. Mental Model

Think of a KG as a **giant pin-board of facts**, where every fact is one arrow:

```
            worksAt                  hqIn          partOf        partOf
   Alice ───────────▶ Acme Corp ───────────▶ Berlin ───────▶ Germany ───────▶ EU
     │                    ▲
     │ knows              │ foundedIn
     ▼                    │
    Bob              "1998"^^xsd:gYear
```

Every arrow is a **triple**: `(subject, predicate, object)` — `(Alice, worksAt, AcmeCorp)`. The
whole graph is just a *set of triples*. There's no table, no fixed columns: add a new kind of fact
by adding a new predicate. That's the entire RDF data model.

Two practical dialects of this idea:
- **RDF triple stores** (rdflib, Blazegraph, GraphDB, Amazon Neptune-RDF): everything is a triple,
  entities are global URIs, queried with **SPARQL**. Strong on standards, shared vocabularies, and
  W3C reasoning (RDFS/OWL).
- **Labelled property graphs** (Neo4j, Memgraph, Neptune-PG): nodes and edges carry key/value
  *properties* directly, queried with **Cypher/Gremlin**. Friendlier for app developers and
  pathfinding; weaker on formal semantics.

Same mental picture — nodes and typed edges — just different bookkeeping for attributes.

## 3. Key Concepts

- **Triple / statement** — the atom of knowledge: `(subject, predicate, object)`. A KG *is* a set
  of triples. Subjects and predicates are identifiers; objects are either other entities or literal
  values.
- **IRI/URI** — a global identifier for an entity or relation (e.g. `http://example.org/Alice`).
  Using IRIs means two datasets that reference the same IRI are talking about the same thing — the
  basis of *linking* data.
- **Literal** — a concrete value (`"1998"`, `42`, a date) with an optional datatype (`xsd:gYear`)
  or language tag (`"Berlin"@en`). Literals can only be objects, never subjects.
- **Namespace / prefix** — a shared URI stem (`ex: = http://example.org/`) so you write `ex:Alice`
  instead of the full IRI.
- **Ontology / schema (RDFS, OWL)** — the vocabulary: which classes and properties exist,
  `subClassOf`/`subPropertyOf` hierarchies, domains and ranges. This is what enables **inference** —
  deriving new triples (if `Researcher subClassOf Person` and `Alice a Researcher`, then
  `Alice a Person`).
- **SPARQL** — the query language for RDF: graph-pattern matching with `SELECT`, plus `CONSTRUCT`,
  `ASK`, property paths (`ex:partOf+` for transitive reach), and federation across endpoints.
- **Reification / qualifiers** — saying something *about* a triple (who asserted it, when, with what
  confidence). RDF-star and Wikidata-style qualifier nodes handle this.
- **Open World Assumption (OWA)** — absence of a triple means *unknown*, not *false* (unlike a SQL
  database's closed world). Critical when reasoning: you can't conclude "Alice has no employer" just
  because no `worksAt` triple exists.

## 4. Setup

Pure-Python, CPU-only. `rdflib` gives you an in-memory triple store plus a SPARQL engine; `networkx`
lets us treat the same data as a property graph for classic graph algorithms.

```bash
pip install rdflib networkx
```

In [ ]:
# %pip install rdflib networkx
import rdflib, networkx as nx
print("rdflib  ", rdflib.__version__)
print("networkx", nx.__version__)

## 5. Worked Examples

We'll build a tiny company/people/geography KG, query it with SPARQL, view it as a property graph
for pathfinding, and finally (gated behind an env var) hit the public Wikidata endpoint.

### Example 1 — Build a knowledge graph from triples

A KG is just a set of `(subject, predicate, object)` triples. We declare a namespace for our own
IRIs, add facts, and serialize to **Turtle** — the human-readable RDF syntax.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, XSD

EX = Namespace("http://example.org/")
g = Graph()
g.bind("ex", EX)

# --- schema: a small class hierarchy + property metadata ---
g.add((EX.Researcher, RDFS.subClassOf, EX.Person))
g.add((EX.worksAt, RDFS.domain, EX.Person))
g.add((EX.worksAt, RDFS.range, EX.Organization))

# --- instance data: entities and the typed edges between them ---
g.add((EX.Alice, RDF.type, EX.Researcher))
g.add((EX.Alice, RDFS.label, Literal("Alice", lang="en")))
g.add((EX.Alice, EX.worksAt, EX.AcmeCorp))
g.add((EX.Alice, EX.knows, EX.Bob))

g.add((EX.Bob, RDF.type, EX.Person))
g.add((EX.Bob, EX.worksAt, EX.Globex))

g.add((EX.AcmeCorp, RDF.type, EX.Organization))
g.add((EX.AcmeCorp, EX.hqIn, EX.Berlin))
g.add((EX.AcmeCorp, EX.foundedIn, Literal("1998", datatype=XSD.gYear)))
g.add((EX.Globex, RDF.type, EX.Organization))
g.add((EX.Globex, EX.hqIn, EX.Munich))

# --- geography: a transitive partOf hierarchy ---
for child, parent in [(EX.Berlin, EX.Germany), (EX.Munich, EX.Germany),
                      (EX.Germany, EX.EU)]:
    g.add((child, EX.partOf, parent))

print(f"{len(g)} triples in the graph\n")
print(g.serialize(format="turtle"))

### Example 2 — Query it with SPARQL

SPARQL is pattern matching over triples: each line in the `WHERE` block is a triple with variables
(`?x`). The engine finds every binding that satisfies all patterns at once. Note the **property
path** `ex:partOf+` — one or more `partOf` hops — which gives us transitive containment for free.

In [ ]:
# Q1: who works where, and in which city is the employer based?
q1 = """
PREFIX ex: <http://example.org/>
SELECT ?person ?org ?city WHERE {
    ?person ex:worksAt ?org .
    ?org    ex:hqIn    ?city .
}"""
print("Who works where:")
for row in g.query(q1):
    name = lambda t: t.split("/")[-1]
    print(f"  {name(row.person):6} -> {name(row.org):9} (HQ in {name(row.city)})")

# Q2: transitive reach via a property path — every region Alice's employer sits inside.
q2 = """
PREFIX ex: <http://example.org/>
SELECT ?region WHERE {
    ex:Alice ex:worksAt/ex:hqIn ?city .
    ?city ex:partOf+ ?region .
}"""
print("\nRegions containing Alice's employer's city:")
for row in g.query(q2):
    print(f"  {row.region.split('/')[-1]}")

### Example 3 — RDFS inference and the property-graph view

Two everyday operations: (a) **inference** — deriving facts the schema entails but nobody stated,
and (b) treating the RDF as a **property graph** so we can run classic algorithms (shortest path,
connectivity) with networkx.

In [ ]:
# (a) Manual RDFS-style inference: Researcher subClassOf Person  =>  Alice a Person.
#     rdflib ships no full reasoner, but transitive_subjects/objects cover the common closures.
inferred = []
for s, _, c in g.triples((None, RDF.type, None)):
    for sup in g.transitive_objects(c, RDFS.subClassOf):
        if sup != c and (s, RDF.type, sup) not in g:
            inferred.append((s, RDF.type, sup))
for t in inferred:
    g.add(t)
print("Inferred type triples:")
for s, _, o in inferred:
    print(f"  {s.split('/')[-1]} a {o.split('/')[-1]}")

# (b) Project the RDF onto a directed networkx graph (entity->entity edges only,
#     dropping literals and type/label metadata) and run graph algorithms.
SKIP = {RDF.type, RDFS.label, RDFS.subClassOf, RDFS.domain, RDFS.range}
G = nx.DiGraph()
for s, p, o in g:
    if p in SKIP or isinstance(o, Literal):
        continue
    G.add_edge(s.split("/")[-1], o.split("/")[-1], rel=p.split("/")[-1])

print(f"\nProperty graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
path = nx.shortest_path(G, "Alice", "EU")
print("Shortest path Alice -> EU:")
print("  " + " -> ".join(path))
print("Most connected node (degree centrality):",
      max(nx.degree_centrality(G).items(), key=lambda kv: kv[1])[0])

### Example 4 — Querying a public KG (Wikidata), network-gated

The same SPARQL skills hit real public KGs. This cell calls Wikidata's live endpoint, so it's gated
behind an env var — set `RUN_WIKIDATA=1` to actually run it; otherwise we just show the query shape
so the notebook still executes top-to-bottom offline.

In [ ]:
import os

# A SPARQL query for cities in Germany with population > 1M, by descending population.
wikidata_query = """
SELECT ?cityLabel ?population WHERE {
  ?city wdt:P31/wdt:P279* wd:Q515 ;   # instance of (a subclass of) city
        wdt:P17 wd:Q183 ;             # country = Germany
        wdt:P1082 ?population .       # population
  FILTER(?population > 1000000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
} ORDER BY DESC(?population)"""

if os.getenv("RUN_WIKIDATA"):
    from SPARQLWrapper import SPARQLWrapper, JSON  # pip install SPARQLWrapper
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql",
                           agent="praxis-notebook/1.0")
    sparql.setQuery(wikidata_query)
    sparql.setReturnFormat(JSON)
    for r in sparql.query().convert()["results"]["bindings"]:
        print(f"  {r['cityLabel']['value']:12} {int(r['population']['value']):>10,}")
else:
    print("RUN_WIKIDATA not set — skipping live call. Query shape:")
    print(wikidata_query)

## 6. Gotchas & Pitfalls

- **Identity / entity resolution is the hard part.** Two sources call the same company "Acme Corp"
  and "ACME Inc." If you don't reconcile them to one IRI, your graph silently splits into
  disconnected islands and multi-hop queries return nothing. Budget real effort for `owl:sameAs`
  linking / dedup.
- **Open World Assumption surprises.** A missing triple means *unknown*, not *false*. `NOT EXISTS`
  in SPARQL queries the data you have, not the truth — "no `worksAt` triple" ≠ "unemployed".
- **IRIs are opaque, not labels.** `ex:Q42` is a perfectly good entity IRI; never parse meaning out
  of an IRI string. Always carry human-readable `rdfs:label`s for display and search.
- **rdflib has no built-in OWL reasoner.** It does basic graph ops and SPARQL; for real
  RDFS/OWL entailment use `owlrl` (`from owlrl import DeductiveClosure`) or a reasoning store like
  GraphDB. Our Example 3 hand-rolled one specific closure on purpose.
- **Unbounded property paths and cartesian patterns explode.** `?a ex:knows+ ?b` over a dense graph,
  or a `WHERE` block with no shared variable between patterns, can blow up combinatorially. Constrain
  with anchors, `LIMIT`, and selective triple patterns first.
- **Blank nodes break round-trips.** Anonymous nodes (`_:b0`) get fresh IDs every parse; don't rely
  on their identity across serializations. Prefer minting real IRIs.
- **Modelling n-ary facts.** A salary that depends on (person, company, year) doesn't fit one triple.
  Use a qualifier/intermediate node (Wikidata statements) or RDF-star — don't cram it into the
  predicate name.
- **Property graph vs RDF mismatch.** Edge properties (a `since` date on `worksAt`) have no native
  home in plain RDF — you must reify. Picking the wrong model for your access pattern is a costly
  rewrite later.

## 7. When to Use vs Alternatives

| Option | Best at | Weak at | Reach for it when |
|---|---|---|---|
| **RDF triple store + SPARQL** (rdflib, GraphDB, Neptune) | Standards, shared vocabularies, federation, OWL/RDFS reasoning | Developer ergonomics; edge properties need reification | Integrating open/linked data, needing formal semantics & inference |
| **Labelled property graph** (Neo4j, Memgraph) | Pathfinding, edge properties, app-developer UX (Cypher) | Formal semantics, cross-dataset linking | App-centric graphs: fraud rings, recommendations, networks |
| **Relational / SQL** | Uniform tabular data, aggregations, transactions, maturity | Deep/variable-hop traversal (recursive joins hurt) | Data is regular rows and queries are analytics, not connections |
| **Document store** (Mongo, Elastic) | Flexible nested records, full-text search | Cross-document relationships and joins | Self-contained documents with little linking |
| **Vector DB / embeddings** | Fuzzy semantic similarity, RAG retrieval | Exact multi-hop logic, explainability | Similarity search; pair *with* a KG for GraphRAG |

**Honest takes.** A KG earns its keep when relationships are the product and the data is messy and
heterogeneous — integration, reasoning, explainable multi-hop answers. If your data is uniform and
your questions are `GROUP BY`, SQL will be faster and simpler; don't graph-ify it for fashion.
RDF vs property graph is mostly about *semantics & interop* (RDF) vs *developer velocity & edge
attributes* (property graph). Increasingly KGs and **LLMs** are complements: the KG supplies
verifiable, structured grounding (GraphRAG) and the LLM supplies natural-language access and
extraction to *populate* the graph. See
[`knowledge-graph-embeddings`](../02-ai-ml-tooling/knowledge-graph-embeddings.ipynb) for the ML angle.

## 8. Resources

- **RDF 1.1 Primer (W3C)** — the canonical, readable intro to the triple model:
  https://www.w3.org/TR/rdf11-primer/
- **SPARQL 1.1 Query spec (W3C)** — the query language, including property paths and federation:
  https://www.w3.org/TR/sparql11-query/
- **rdflib documentation** — the Python library used here (graphs, namespaces, SPARQL):
  https://rdflib.readthedocs.io/
- **Wikidata SPARQL query service + examples** — a massive live KG to practice on:
  https://query.wikidata.org/ (example gallery under the "?" menu)
- **Neo4j — Graph Databases (free O'Reilly book)** — the property-graph perspective and modelling
  patterns: https://neo4j.com/graph-databases-book/
- **"Knowledge Graphs" (Hogan et al., ACM Computing Surveys 2021)** — the comprehensive academic
  survey of the whole field: https://arxiv.org/abs/2003.02320

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def entailed_types(triples):
    """The type triples the class hierarchy entails but nobody stated."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE